# Loading dataframe and defining features and target

In [4]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix
import lightgbm as lgb
from lightgbm import early_stopping
import time
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore")

SavePlots = False

data = pd.DataFrame(np.genfromtxt('data_mice_encoded_aggr.csv', delimiter=',', names=True))

target = 'CLAIM_COUNT'

features = [
    col for col in data.columns
    if col not in ['CLAIM_COUNT', 'CLAIM_SIZE', 'CLAIM_SIZE_INDEX', 
                   'f0', 'POLICY', 'OUTER_WALLS', 'HEATING_TYPE', 
                   'WATER_SUPPLY_TYPE', 'ROOF_TYPE', 'CLAIM_RATE']
]

data = data[data["EXPOSURE"] > 0]

X = data[features]
y = data[target]

feature_names = np.array(X.columns)

## Defining candidates for final feature set

In [7]:
top20_perm = ['EXPOSURE', 'DEDUCTIBLE', 'RESIDENTIAL_AREA', 'HEATING_TYPE_CAT_District_Heating', 'CONSTRUCTION_YEAR', 'HOUDEN10KM', 'WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant', 'HEATING_TYPE_CAT_Central_Heating_Two_Units', 'CONSERVATORY_AREA', 'BUILDINGS', 'HEATING_TYPE_CAT_Stove_Fireplace', 'HEATING_TYPE_CAT_Heat_Pump', 'WATER_SUPPLY_TYPE_CAT_Non_Public_Water_Supply', 'BASEMENT_AREA', 'WETROOMS', 'ROOF_TYPE_CAT_Concrete_Tiles', 'ROOF_TYPE_CAT_Fiber_Cement_Asbestos', 'ROOF_TYPE_CAT_Brick', 'ROOF_TYPE_CAT_Fiber_Cement', 'OUTER_WALLS_CAT_Lightweight_Concrete']

cluster_representatives = ['WATER_SUPPLY_TYPE_CAT_Private_Water_Supply', 'HEATING_TYPE_CAT_Unknown', 'WATER_SUPPLY_TYPE_CAT_No_Water_Supply', 'ROOF_TYPE_CAT_Glass', 'HEATING_TYPE_CAT_Gas_Radiators', 'OUTER_WALLS_CAT_Fiber_Cement', 'WATER_SUPPLY_TYPE_CAT_Mixed_Water_Supply', 'HEATING_TYPE_CAT_Mixed', 'OUTER_WALLS_CAT_Fiber_Cement_Plates', 'OUTER_WALLS_CAT_Glass', 'OUTER_WALLS_CAT_Lightweight_Concrete', 'WATER_SUPPLY_TYPE_CAT_Well_Water', 'OUTER_WALLS_CAT_Other_Material', 'CONSERVATORY_AREA', 'FLOORS', 'OUTER_WALLS_CAT_Concrete_Elements', 'OUTER_WALLS_CAT_Metal_Plates', 'ROOF_TYPE_CAT_Thatched_Roof', 'ROOF_TYPE_CAT_Flat_Roof', 'ROOF_TYPE_CAT_Other_Material', 'ROOF_TYPE_CAT_PVC', 'ROOF_TYPE_CAT_Fiber_Cement', 'HEATING_TYPE_CAT_District_Heating', 'HEATING_TYPE_CAT_Central_Heating_Own_System', 'HEATING_TYPE_CAT_Stove_Fireplace', 'ROOF_TYPE_CAT_Metal_Sheets', 'OUTER_WALLS_CAT_Brick_Walls', 'HEATING_TYPE_CAT_Heat_Pump', 'EXPOSURE', 'ROOF_TYPE_CAT_Tar_Paper', 'ROOF_TYPE_CAT_Brick', 'WATER_SUPPLY_TYPE_CAT_Non_Public_Water_Supply', 'BASEMENT_AREA', 'WETROOMS', 'HEATING_TYPE_CAT_No_Heating', 'WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant', 'HEATING_TYPE_CAT_Electric_Heating', 'OUTER_WALLS_CAT_Wood_Cladding', 'CONSTRUCTION_YEAR', 'ROOF_TYPE_CAT_Concrete_Tiles', 'OUTER_WALLS_CAT_Timber_Framing', 'HEATING_TYPE_CAT_Central_Heating_Two_Units', 'HOUDEN10KM', 'BUILDINGS', 'ROOF_TYPE_CAT_Fiber_Cement_Asbestos', 'RESIDENTIAL_AREA', 'DEDUCTIBLE']

top20 = ['EXPOSURE', 'RESIDENTIAL_AREA', 'DEDUCTIBLE', 'CONSTRUCTION_YEAR', 'HOUDEN10KM', 'HEATING_TYPE_CAT_District_Heating', 'BUILDINGS', 'CONSERVATORY_AREA', 'BASEMENT_AREA', 'WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant', 'HEATING_TYPE_CAT_Central_Heating_Two_Units', 'HEATING_TYPE_CAT_Heat_Pump', 'WETROOMS', 'ROOF_TYPE_CAT_Fiber_Cement_Asbestos', 'HEATING_TYPE_CAT_Stove_Fireplace', 'ROOF_TYPE_CAT_Brick', 'WATER_SUPPLY_TYPE_CAT_Non_Public_Water_Supply', 'ROOF_TYPE_CAT_Concrete_Tiles', 'HEATING_TYPE_CAT_Central_Heating_Own_System', 'ROOF_TYPE_CAT_Fiber_Cement']

top20_consensus = ['EXPOSURE', 'RESIDENTIAL_AREA', 'DEDUCTIBLE', 'CONSTRUCTION_YEAR', 'HOUDEN10KM', 'HEATING_TYPE_CAT_District_Heating', 'BUILDINGS', 'CONSERVATORY_AREA', 'BASEMENT_AREA', 'WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant', 'HEATING_TYPE_CAT_Central_Heating_Two_Units', 'HEATING_TYPE_CAT_Heat_Pump', 'WETROOMS', 'ROOF_TYPE_CAT_Fiber_Cement_Asbestos', 'HEATING_TYPE_CAT_Stove_Fireplace', 'ROOF_TYPE_CAT_Brick', 'WATER_SUPPLY_TYPE_CAT_Non_Public_Water_Supply', 'ROOF_TYPE_CAT_Concrete_Tiles', 'HEATING_TYPE_CAT_Central_Heating_Own_System', 'ROOF_TYPE_CAT_Fiber_Cement']

top20_stability = ['EXPOSURE', 'DEDUCTIBLE', 'CONSTRUCTION_YEAR', 'HEATING_TYPE_CAT_District_Heating', 'RESIDENTIAL_AREA', 'HOUDEN10KM', 'WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant', 'HEATING_TYPE_CAT_Heat_Pump', 'BUILDINGS', 'BASEMENT_AREA', 'HEATING_TYPE_CAT_Central_Heating_Two_Units', 'HEATING_TYPE_CAT_Stove_Fireplace', 'CONSERVATORY_AREA', 'WATER_SUPPLY_TYPE_CAT_Non_Public_Water_Supply', 'ROOF_TYPE_CAT_Tar_Paper', 'WETROOMS', 'ROOF_TYPE_CAT_Concrete_Tiles', 'ROOF_TYPE_CAT_Fiber_Cement_Asbestos', 'WATER_SUPPLY_TYPE_CAT_Mixed_Water_Supply', 'WATER_SUPPLY_TYPE_CAT_Well_Water']

data_perm = data[top20_perm + ["CLAIM_COUNT"]]
data_clust_rep = data[cluster_representatives + ["CLAIM_COUNT"]]
data_20 = data[top20 + ["CLAIM_COUNT"]]
data_20_consensus = data[top20_consensus + ["CLAIM_COUNT"]]
data_20_stab = data[top20_stability + ["CLAIM_COUNT"]]

groups = {
    "perm": data[top20_perm],
    "clust_rep": data[cluster_representatives],
    "top20": data[top20],
    "consensus": data[top20_consensus],
    "stability": data[top20_stability]
}

# Initial HP grid search

## Tree-based models

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split


np.random.seed(42)


TARGETS = {
    "raw": data["CLAIM_COUNT"].values,
    "log": np.log1p(data["CLAIM_COUNT"].values)
}


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


LGBM_GRID = [
    {"num_leaves": 31, "learning_rate": 0.05},
    {"num_leaves": 63, "learning_rate": 0.05},
    {"num_leaves": 63, "learning_rate": 0.1},
]

XGB_GRID = [
    {"max_depth": 4, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.1},
]

CAT_GRID = [
    {"depth": 6, "learning_rate": 0.05},
    {"depth": 8, "learning_rate": 0.05},
    {"depth": 6, "learning_rate": 0.1},
]

n_seeds = 5

def build_lgbm(seed, params):
    return lgb.LGBMRegressor(
        objective="regression",
        n_estimators=5000,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbose=-1,
        **params
    )


def build_xgb(seed, params):
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=5000,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="rmse",
        random_state=seed,
        n_jobs=-1,
        **params
    )


def build_cat(seed, params):
    return CatBoostRegressor(
        loss_function="RMSE",
        iterations=5000,
        subsample=0.8,
        random_seed=seed,
        verbose=0,
        task_type="CPU",
        **params
    )


MODELS = {
    "LightGBM": (build_lgbm, LGBM_GRID),
    "XGBoost": (build_xgb, XGB_GRID),
    "CatBoost": (build_cat, CAT_GRID)
}

results = {}

for target_name, y_full in TARGETS.items():

    print(f"\n\n##############################")
    print(f"TARGET: {target_name}")
    print(f"##############################")

    results[target_name] = {}

    for model_name, (builder, grid) in MODELS.items():

        print(f"\n==============================")
        print(f"MODEL: {model_name}")
        print(f"==============================")

        results[target_name][model_name] = []

        for config_id, params in enumerate(grid):

            print(f"\n>>> Config {config_id+1}/{len(grid)}: {params}")

            config_results = {}

            for group_name, X_full in groups.items():

                print(f"  -> Group: {group_name}")

                losses = []

                for seed in range(n_seeds):

                    X_train, X_valid, y_train, y_valid = train_test_split(
                        X_full,
                        y_full,
                        test_size=0.25,
                        random_state=seed
                    )

                    model = builder(seed, params)

                    if model_name == "LightGBM":
                        model.fit(
                            X_train,
                            y_train,
                            eval_set=[(X_valid, y_valid)],
                            callbacks=[lgb.early_stopping(50, verbose=False)]
                        )

                    elif model_name == "XGBoost":
                        model.fit(
                            X_train,
                            y_train,
                            eval_set=[(X_valid, y_valid)],
                            verbose=False
                        )

                    elif model_name == "CatBoost":
                        model.fit(
                            X_train,
                            y_train,
                            eval_set=(X_valid, y_valid),
                            early_stopping_rounds=50
                        )

                    pred = model.predict(X_valid)

                    if target_name == "log":
                        pred_eval = np.expm1(pred)
                        y_eval = np.expm1(y_valid)
                    else:
                        pred_eval = pred
                        y_eval = y_valid

                    loss = rmse(y_eval, pred_eval)
                    losses.append(loss)

                config_results[group_name] = np.mean(losses)

            rank_score = np.mean(list(config_results.values()))

            results[target_name][model_name].append({
                "params": params,
                "scores": config_results,
                "rank_score": rank_score
            })

rows = []

for target_name in results:
    for model_name in results[target_name]:
        for cfg in results[target_name][model_name]:

            params = cfg["params"]
            scores = cfg["scores"]

            for feature_set, score in scores.items():
                rows.append({
                    "target": target_name,
                    "model": model_name,
                    "feature_set": feature_set,
                    "score": score,
                    **params
                })

df_results = pd.DataFrame(rows)

df_sorted = df_results.sort_values("score", ascending=True)

print("\n===== FINAL RESULTS =====")
print(df_sorted)

## NN (MLPRegressor) initial HP grid search

In [ ]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

firedata = data.copy()

TARGETS = {
    "raw": y.copy(),
    "log": np.log1p(y.copy())
}


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


MLP_GRID = [
    {
        "hidden_layer_sizes": (64, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },
    {
        "hidden_layer_sizes": (128, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },
    {
        "hidden_layer_sizes": (128, 128),
        "alpha": 1e-3,
        "learning_rate_init": 5e-4
    }
]

n_seeds = 10


def build_mlp(seed, params):

    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            max_iter=800,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=seed,
            **params
        ))
    ])


results = []

print("\n==============================")
print("MLP MODEL SEARCH")
print("==============================")

for target_name, y_full in TARGETS.items():

    print(f"\n\n##############################")
    print(f"TARGET: {target_name}")
    print(f"##############################")

    for config_id, params in enumerate(MLP_GRID):

        print(f"\n>>> Config {config_id+1}/{len(MLP_GRID)}")
        print(params)

        for group_name, X_full in groups.items():

            print(f"\n  -> Feature set: {group_name}")

            losses = []

            for seed in tqdm(range(n_seeds),
                             desc=f"{target_name} | Config {config_id+1} | {group_name}",
                             leave=False):

                X_train, X_valid, y_train, y_valid = train_test_split(
                    X_full,
                    y_full,
                    test_size=0.25,
                    random_state=seed
                )

                model = build_mlp(seed, params)

                model.fit(X_train, y_train)

                pred = model.predict(X_valid)

                if target_name == "log":
                    pred_eval = np.expm1(pred)
                    y_eval = np.expm1(y_valid)
                else:
                    pred_eval = pred
                    y_eval = y_valid

                loss = rmse(y_eval, pred_eval)
                losses.append(loss)

            mean_loss = np.mean(losses)

            results.append({
                "target": target_name,
                "model": "MLP",
                "feature_set": group_name,
                "score": mean_loss,
                **params
            })

            print(f"     RMSE = {mean_loss:.6f}")


df_mlp = pd.DataFrame(results)

df_mlp_sorted = (
    df_mlp
    .sort_values("score", ascending=True)
    .reset_index(drop=True)
)

print("\n===== FINAL MLP RESULTS =====")
print(df_mlp_sorted)

In [11]:
df_lgbm = df_sorted[df_sorted["model"] == "LightGBM"].copy() 
df_lgbm = df_lgbm.drop(columns=["max_depth", "depth"], errors="ignore") 
print(df_lgbm.head(3))

df_xgb = df_sorted[df_sorted["model"] == "XGBoost"].copy() 
df_xgb = df_xgb.drop(columns=["num_leaves", "depth"], errors="ignore") 
print(df_xgb.head(3))

df_cat = df_sorted[df_sorted["model"] == "CatBoost"].copy() 
df_cat = df_cat.drop(columns=["max_depth", "num_leaves"], errors="ignore") 
print(df_cat.head(3))

print(df_mlp_sorted.head(3))

  target     model feature_set     score  num_leaves  learning_rate
4    raw  LightGBM   stability  0.138292        31.0           0.05
3    raw  LightGBM   consensus  0.138295        31.0           0.05
2    raw  LightGBM       top20  0.138295        31.0           0.05
   target    model feature_set     score  learning_rate  max_depth
64    log  XGBoost   stability  0.139422           0.05        4.0
62    log  XGBoost       top20  0.139465           0.05        4.0
63    log  XGBoost   consensus  0.139465           0.05        4.0
   target     model feature_set     score  learning_rate  depth
34    raw  CatBoost   stability  0.138221           0.05    6.0
44    raw  CatBoost   stability  0.138232           0.10    6.0
30    raw  CatBoost        perm  0.138232           0.05    6.0
  target model feature_set     score hidden_layer_sizes   alpha  \
0    raw   MLP       top20  0.139437          (128, 64)  0.0001   
1    raw   MLP   consensus  0.139437          (128, 64)  0.0001   
2  

# Final HP optimization

Given the best feature set, which hyperparameter optimization strategy (Grid, Random, or Optuna) finds the best CatBoost hyperparameters, and how long does each take?

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterSampler
from scipy.stats import randint, loguniform

import optuna
from catboost import CatBoostRegressor


winner = df_cat.sort_values("score").iloc[0]

feature_map = {
    "perm": top20_perm,
    "clust_rep": cluster_representatives,
    "top20": top20,
    "consensus": top20_consensus,
    "stability": top20_stability
}

best_feature_set = winner["feature_set"]

X_full = data[feature_map[best_feature_set]].astype(np.float32)

y_full = data["CLAIM_COUNT"].values

n_seeds = 3

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def eval_catboost(params):

    losses = []

    for seed in range(n_seeds):

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_full,
            test_size=0.25,
            random_state=seed
        )

        model = CatBoostRegressor(
            iterations=3000,
            loss_function="RMSE",
            random_seed=seed,
            verbose=0,
            early_stopping_rounds=50,
            **params
        )

        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid)
        )

        pred = model.predict(X_valid)

        losses.append(
            rmse(y_valid, pred)
        )

    return np.mean(losses)

grid = {
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 10]
}

grid_params = [
    dict(zip(grid.keys(), v))
    for v in np.array(
        np.meshgrid(*grid.values())
    ).T.reshape(-1, len(grid))
]

start = time.time()

grid_results = []

for i, params in enumerate(grid_params, start=1):

    print(f"Grid {i}/{len(grid_params)}")

    score = eval_catboost(params)

    grid_results.append({
        **params,
        "score": score
    })

grid_time = time.time() - start

df_grid = (
    pd.DataFrame(grid_results)
    .sort_values("score")
    .reset_index(drop=True)
)

random_space = {
    "depth": randint(4, 10),
    "learning_rate": loguniform(0.01, 0.2),
    "l2_leaf_reg": randint(1, 20)
}

random_params = list(
    ParameterSampler(
        random_space,
        n_iter=20,
        random_state=42
    )
)

start = time.time()

random_results = []

for i, params in enumerate(random_params, start=1):

    print(f"Random {i}/{len(random_params)}")

    score = eval_catboost(params)

    random_results.append({
        **params,
        "score": score
    })

random_time = time.time() - start

df_random = (
    pd.DataFrame(random_results)
    .sort_values("score")
    .reset_index(drop=True)
)

def objective(trial):

    params = {

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg",
            1,
            20
        )
    }

    return eval_catboost(params)


start = time.time()

study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

optuna_time = time.time() - start

comparison = pd.DataFrame([
    {
        "method": "Grid",
        "best_score": df_grid["score"].min(),
        "runtime_sec": grid_time
    },
    {
        "method": "Random",
        "best_score": df_random["score"].min(),
        "runtime_sec": random_time
    },
    {
        "method": "Optuna",
        "best_score": study.best_value,
        "runtime_sec": optuna_time
    }
]).sort_values("best_score")


print("\n===== FINAL COMPARISON =====")
print(comparison)

print("\n===== BEST GRID PARAMS =====")
print(df_grid.iloc[0])

print("\n===== BEST RANDOM PARAMS =====")
print(df_random.iloc[0])

print("\n===== BEST OPTUNA PARAMS =====")
print(study.best_params)

### Final validation

In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor


best_params = {
    "depth": 6,
    "l2_leaf_reg": 14,
    "learning_rate": 0.032269
}


n_seeds = 10
seed_scores = []

print("\n==============================")
print("FINAL ROBUST VALIDATION")
print("==============================")

print("\nBest parameters:")
print(best_params)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


for seed in range(n_seeds):

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_full,
        y_full,
        test_size=0.25,
        random_state=seed
    )

    model = CatBoostRegressor(
        iterations=3000,
        loss_function="RMSE",
        random_seed=seed,
        verbose=0,
        early_stopping_rounds=50,
        **best_params
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid)
    )

    pred = model.predict(X_valid)

    score = rmse(y_valid, pred)
    seed_scores.append(score)

    print(
        f"Seed {seed:2d}: "
        f"RMSE = {score:.6f}"
    )


seed_scores = np.array(seed_scores)

summary = pd.DataFrame({
    "seed": np.arange(n_seeds),
    "score": seed_scores
})

print("\n===== SEED RESULTS =====")
print(summary)

print("\n===== FINAL ESTIMATE =====")

print(f"Mean RMSE : {seed_scores.mean():.6f}")
print(f"Std RMSE  : {seed_scores.std(ddof=1):.6f}")
print(f"Min RMSE  : {seed_scores.min():.6f}")
print(f"Max RMSE  : {seed_scores.max():.6f}")


final_model = CatBoostRegressor(
    iterations=int(1.2 * 3000),
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    **best_params
)

final_model.fit(
    X_full,
    y_full
)

print("\nFinal model fitted on full dataset.")


FINAL ROBUST VALIDATION

Best parameters:
{'depth': 6, 'l2_leaf_reg': 14, 'learning_rate': 0.032269}
Seed  0: RMSE = 0.137627
Seed  1: RMSE = 0.140015
Seed  2: RMSE = 0.138937
Seed  3: RMSE = 0.134830
Seed  4: RMSE = 0.139655
Seed  5: RMSE = 0.136778
Seed  6: RMSE = 0.139941
Seed  7: RMSE = 0.142638
Seed  8: RMSE = 0.140119
Seed  9: RMSE = 0.141341

===== SEED RESULTS =====
   seed     score
0     0  0.137627
1     1  0.140015
2     2  0.138937
3     3  0.134830
4     4  0.139655
5     5  0.136778
6     6  0.139941
7     7  0.142638
8     8  0.140119
9     9  0.141341

===== FINAL ESTIMATE =====
Mean RMSE : 0.139188
Std RMSE  : 0.002265
Min RMSE  : 0.134830
Max RMSE  : 0.142638

Final model fitted on full dataset.
